# 08 · Figures

This notebook draws the ten figures of the paper: Figs. 1–4 and Supplementary Figs. 1–6. Every plotted value is read from the outcome
tables of notebooks 04–06 (`SUMMARY`) and, for Fig. 1, from the counts of the cohort (`COHORT_SUMMARY`, notebook 02); nothing is
computed from patient-level data here. Each figure is saved in `FIGURES` as a PNG at 300 dpi, a PDF and `<name>_source.csv` with the
plotted values (source data), under the names `Fig1` … `Fig4` and `SuppFig1` … `SuppFig6`. Positions and sizes are in millimetres on the
180-mm width of the journal (display style in notebook 00).

The notebook runs in about 10 seconds on a CPU.

In [ ]:
%run ./00_setup_and_settings.ipynb

## Settings and inputs

`OVERWRITE = False` keeps figures whose files exist in `FIGURES`; set it to `True` to draw every figure again. Histogram bins that hold
fewer than 10 patients are removed here, so that they can never be drawn.

In [ ]:
import json

from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D
from matplotlib.patches import Arc, FancyBboxPatch, Patch
from matplotlib.ticker import FormatStrFormatter

OVERWRITE = False
print("font:", setup_figures())

cohort_summary = json.loads(COHORT_SUMMARY.read_text(encoding="utf-8"))
accuracy = pd.read_csv(SUMMARY / "accuracy.csv").set_index("key")
reliability = pd.read_csv(SUMMARY / "reliability.csv").set_index(["key", "set"])
hist = pd.read_csv(SUMMARY / "difference_hist.csv")
hist.loc[~hist["shown"].astype(bool), "pct"] = np.nan
decomposition = pd.read_csv(SUMMARY / "decomposition.csv").set_index(["key", "component"])
night_day = pd.read_csv(SUMMARY / "night_day.csv").set_index("key")
slope = pd.read_csv(SUMMARY / "slope.csv").set_index("key")
slope_by_age = pd.read_csv(SUMMARY / "slope_by_age.csv")
mortality = pd.read_csv(SUMMARY / "mortality.csv").set_index("key")
simex = pd.read_csv(SUMMARY / "simex_curve.csv")
averaging = pd.read_csv(SUMMARY / "averaging.csv")
flags = pd.read_csv(SUMMARY / "flags.csv").set_index("key")

## Shared drawing functions

The color stands for the foundation model and the marker for the adaptation (`FM_COLOR`, `MARKER` in notebook 00), in every figure.
Figures with one row per model list the models in the order of `MODEL_KEYS`, top to bottom. Figures with one small panel per foundation
model draw the adaptations with different line styles: dotted for frozen, solid for fine-tuned on MIMIC-IV-ECG, dashed for fine-tuned on
CODE-15%. The axis limits are fixed; `warn_if_clipped` reports any plotted value that falls outside them.

In [ ]:
LABELS = [model_label(k) for k in MODEL_KEYS]
FM_PANELS = [("Lima", "Lima et al. model"), ("ECGFounder", "ECGFounder"), ("ECG-FM", "ECG-FM"), ("HuBERT-ECG", "HuBERT-ECG")]
LINE_STYLE = {"public": "-", "frozen": (0, (1, 1)), "mimic": "-", "code15": (0, (4, 1.5))}
OR_LABEL = "Odds ratio for death within 1 year\nper 10-year larger age gap"


def colour(key):
    return FM_COLOR[MODELS.fm[key]]


def marker(key):
    return MARKER[MODELS.adapt[key]]


def keys_of(fm):
    return [k for k in MODEL_KEYS if MODELS.fm[k] == fm]


def model_legend(fig, y_mm=1.5):
    """Legend along the bottom of a figure: colour = foundation model, marker = adaptation."""
    h = [Line2D([], [], ls="none", marker=MARKER["public"], color=FM_COLOR["Lima"], ms=3.4, label="Lima et al. model")]
    h += [Line2D([], [], ls="none", marker="o", color=FM_COLOR[f], ms=3.4, label=f) for f in ("ECGFounder", "ECG-FM", "HuBERT-ECG")]
    h += [Line2D([], [], ls="none", marker=MARKER[a], color=GREY, ms=3.4, label=ADAPTATIONS[a]) for a in ("frozen", "mimic", "code15")]
    fig.legend(handles=h, loc="lower center", bbox_to_anchor=(0.5, y_mm / (fig.get_size_inches()[1] / MM)), ncol=7, fontsize=5.5,
               handletextpad=0.3, columnspacing=1.2, borderaxespad=0, frameon=False)


def model_rows(ax, show=True, separators=(1, 4, 7)):
    """One row per model (top to bottom in the order of MODEL_KEYS); separators between the foundation models."""
    ax.set_yticks(range(len(LABELS)))
    ax.set_yticklabels(LABELS if show else [""] * len(LABELS))
    ax.set_ylim(len(LABELS) - 0.5, -0.5)
    ax.tick_params(axis="y", length=0)
    for j in separators:
        ax.axhline(j - 0.5, color=GRID, lw=0.5, zorder=0)


def ci_point(ax, x, y, lo, hi, key, filled=True, ms=3.4):
    """Estimate with a horizontal 95% CI, in the colour and marker of the model (open marker: filled=False)."""
    c = colour(key)
    ax.errorbar([x], [y], xerr=[[x - lo], [hi - x]], fmt=marker(key), ms=ms, color=c, mfc=c if filled else "white", mec=c, mew=0.7,
                ecolor=c, elinewidth=0.7, zorder=3)


def panel_title(ax, fm, title):
    ax.text(0.5, 1.03, title, transform=ax.transAxes, ha="center", va="bottom", fontsize=5.8, fontweight="bold",
            color=FM_COLOR[fm] if fm != "Lima" else INK)


def warn_if_clipped(name, *axes):
    """The axis limits are fixed so that every figure looks the same in each run. Warn when plotted data (for example the end of a
    confidence interval) reach beyond them, so that the limits can be widened instead of cutting the data off."""
    for ax in axes:
        (x0, y0), (x1, y1) = ax.dataLim.get_points()
        (xa, xb), (ya, yb) = sorted(ax.get_xlim()), sorted(ax.get_ylim())
        if x0 < xa or x1 > xb or y0 < ya or y1 > yb:
            print(f"WARNING {name}: plotted values ({x0:.3g} to {x1:.3g}, {y0:.3g} to {y1:.3g}) reach beyond the axis limits "
                  f"({xa:.3g} to {xb:.3g}, {ya:.3g} to {yb:.3g})")


def or_axis(ax, lo, hi, step, label=True):
    """y axis of odds ratios from lo to hi, ticks every `step`; the label only on the first panel of a row."""
    ax.set_ylim(lo, hi)
    ax.set_yticks(np.round(np.arange(np.ceil(lo / step) * step, hi + 1e-9, step), 2))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    if label:
        ax.set_ylabel(OR_LABEL)
    else:
        ax.set_yticklabels([])

## Fig. 1

**Fig. 1 | Study design.** **a**, Example: two ECGs of the same patient recorded 11 min apart can give different AI-ECG ages and age gaps,
and thus different odds ratios (OR) for death within 1 year (schematic tracings and illustrative values; the ORs follow from the OR per
10-year larger age gap of the Lima et al. model in this study). **b**, ECGs, exclusions, linkage and the patient-level split. **c**, The
ten models: the public model of Lima et al. and three ECG foundation models, each used as a frozen embedding with ridge regression,
fine-tuned on MIMIC-IV-ECG or fine-tuned on CODE-15%. The colors and symbols are used in all figures. **d**, Repeat ECG sets within the
test split. Arcs connect the pairs that were analyzed, the time axes are not to scale, and the median interval is weighted so that each
patient contributes equally. The 48-hour set (highlighted) was used for the main analyses. **e**, The three questions of the study, with
the measures and the figure that answers each. Val., validation.

Fig. 1 is a diagram drawn on one canvas in millimetres (y downward). The numbers come from `COHORT_SUMMARY` (panel b), the repeat-ECG sets
in `SUMMARY/reliability.csv` (panel d; the sets are the same for every model) and the OR of the Lima et al. model (panel a). The tracings
of panel a are synthetic.

In [ ]:
GREY_FC, GREY_EC, BLUE_FC, BLUE_EC = "#f6f6f3", "#c3c2b7", "#eaf2fb", "#2a78d6"     # box fills and edges; blue = what the analyses use
SPLIT_FC = {"train": "#e4e3dc", "val": "#f1f0ea", "test": BLUE_FC}


def canvas(H, W=FIGURE_WIDTH):
    fig = figure_mm(H, W)
    A = fig.add_axes([0, 0, 1, 1])
    A.set_xlim(0, W)
    A.set_ylim(H, 0)
    A.axis("off")
    return fig, A


def box(A, x, y, w, h, fc=GREY_FC, ec=GREY_EC, lw=0.6, z=1):
    A.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="square,pad=0", fc=fc, ec=ec, lw=lw, zorder=z))


def text(A, x, y, s, size=5.4, weight="normal", color=None, ha="left", va="top", **kw):
    A.text(x, y, s, fontsize=size, fontweight=weight, color=color or INK, ha=ha, va=va, linespacing=1.25, zorder=5, **kw)


def arrow(A, x0, y0, x1, y1):
    A.annotate("", xy=(x1, y1), xytext=(x0, y0), zorder=4,
               arrowprops=dict(arrowstyle="-|>", lw=0.6, color=AXIS, mutation_scale=5, shrinkA=0, shrinkB=0))


def symbol(A, x, y, adapt, col, ms=4.2):
    A.plot([x], [y], ls="none", marker=MARKER[adapt], ms=ms, color=col, mec="white", mew=0.3, zorder=6)


def heading(A, x, y, letter, title):
    A.text(x, y, letter, fontsize=8, fontweight="bold", ha="left", va="top", zorder=5)
    text(A, x + 4, y + 0.5, title, size=5.8, weight="bold")


def beat(A, x, y, w=3.0, h=2.2, col=None, lw=0.6):
    """One schematic heartbeat (P wave, QRS complex, T wave) with its baseline at y."""
    px = np.array([0, .18, .24, .30, .38, .44, .48, .53, .58, .66, .76, .86, 1.0])
    py = np.array([0, 0, .14, 0, 0, -.12, 1.0, -.32, 0, 0, .3, 0, 0])
    A.plot(x + px * w, y - py * h, color=col or INK, lw=lw, solid_joinstyle="round", zorder=6)


def ecg_strip(A, x, y, w, h, seed, shift=0.0, amp=1.0):
    """Synthetic 3-s tracing (sum of Gaussian waves plus a little noise) on a faint ECG paper grid."""
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 3.0, 900)
    v = np.zeros_like(t)
    for c in np.arange(0.35 + shift, 3.0, 0.86):
        for a, mu, sd in ((.12, -.19, .028), (-.12, -.03, .009), (1.0, 0, .011), (-.28, .028, .011), (.3, .25, .05)):
            v += amp * a * np.exp(-0.5 * ((t - c - mu) / sd) ** 2)
    v += 0.02 * rng.standard_normal(len(t)) + 0.05 * np.sin(2 * np.pi * 0.3 * t + seed)
    A.plot(x + t / 3.0 * w, y + h * 0.62 - v * h * 0.55, color=INK, lw=0.55, zorder=5)
    for gx in np.arange(x, x + w + 0.01, w / 15):
        A.plot([gx, gx], [y, y + h], color="#f3c9c4", lw=0.25, zorder=2)
    for gy in np.arange(y, y + h + 0.01, h / 4):
        A.plot([x, x + w], [gy, gy], color="#f3c9c4", lw=0.25, zorder=2)


def timeline(A, x0, x1, y, times, tmax, col, tick_labels):
    """Schematic time axis with an ECG (heartbeat) at each time and an arc for every pair."""
    A.plot([x0, x1], [y, y], color=AXIS, lw=0.5, zorder=3)
    xs = [x0 + (x1 - x0) * t / tmax for t in times]
    for i in range(len(xs)):
        for j in range(i + 1, len(xs)):
            d = xs[j] - xs[i]
            A.add_patch(Arc(((xs[i] + xs[j]) / 2, y - 0.4), d, min(2.2 + d * 0.18, 6.0), theta1=180, theta2=360, color=col, lw=0.55,
                            alpha=0.75, zorder=4))
    for x in xs:
        A.plot([x, x], [y - 0.6, y + 0.6], color=INK, lw=0.6, zorder=5)
        beat(A, x - 1.3, y - 0.9, w=2.6, h=1.6, col=INK, lw=0.5)
    for t, lab in tick_labels:
        text(A, x0 + (x1 - x0) * t / tmax, y + 1.0, lab, size=4.6, color=INK2, ha="left" if t == 0 else "right")


def example_panel(A, or10, chrono=63, gaps=(12, 3)):
    """a: two ECGs of one patient minutes apart with illustrative AI-ECG ages, and the OR implied by each age gap."""
    text(A, 5, 6.0, f"Chronological age {chrono} years", size=4.8, color=INK2)
    for i, (gap, seed, shift, amp) in enumerate(((gaps[0], 1, 0.0, 1.0), (gaps[1], 2, 0.12, 0.93))):
        yy = 9.3 + i * 13.0
        ecg_strip(A, 5, yy, 28, 10, seed, shift, amp)
        arrow(A, 34.0, yy + 5, 37.5, yy + 5)
        box(A, 38, yy + 0.3, 21, 10.4, fc="white", ec=GREY_EC, lw=0.5)
        for j, (lab, val) in enumerate((("AI-ECG age", f"{chrono + gap} years"), ("Age gap", f"+{gap} years"),
                                        ("Adjusted OR*", f"{or10 ** (gap / 10):.2f}"))):     # OR of the whole gap against a gap of 0
            text(A, 39.2, yy + 1.4 + j * 3.1, lab, size=4.4, color=INK2)
            text(A, 58.0, yy + 1.2 + j * 3.1, val, size=5.0, weight="bold", ha="right")
    text(A, 5, 34.6, f"Example values. *Odds ratio for death within 1 year against an age gap\nof 0, adjusted for age, sex and the number of "
         f"ECGs (OR {or10:.2f} per\n10-year larger age gap, Lima et al. model).", size=4.3, color=INK2)


def cohort_panel(A, x, y, w):
    """b: records, exclusions, linkage and the patient-level split (bar widths proportional to the split)."""
    exc = cohort_summary["excluded_by_first_reason"]
    n_qc = exc["flat_lead"] + exc["missing_lead"]
    n_age = exc["age_89_or_older"] + exc["age_unavailable"] + exc["age_under_18"]
    cx = x + w / 2
    box(A, x, y, w, 8)
    text(A, cx, y + 1.3, "MIMIC-IV-ECG v1.0", weight="bold", ha="center")
    text(A, cx, y + 4.2, f"{cohort_summary['records']:,} ECGs · 2008–2019", ha="center")
    arrow(A, cx, y + 8, cx, y + 15)
    text(A, cx + 1.5, y + 11.5, f"Excluded: flat or missing\nlead ({n_qc:,}); age outside\n18–88 years or unknown ({n_age:,})", size=4.3,
         color=INK2, va="center")
    y2 = y + 15
    box(A, x, y2, w, 10)
    text(A, cx, y2 + 1.3, f"{cohort_summary['included']:,} ECGs · {cohort_summary['patients_included']:,} patients", weight="bold", ha="center")
    text(A, cx, y2 + 4.2, "Linked to MIMIC-IV and MIMIC-IV-ED: care setting,\nage, sex and death within 1 year of each ECG", size=5.0, ha="center")
    arrow(A, cx, y2 + 10, cx, y2 + 14)
    yb, bh, xx = y2 + 15.5, 10.4, x
    text(A, x, yb - 0.3, "Patient-level split", size=5.0, weight="bold", va="bottom")
    ecgs, patients = cohort_summary["split_records"], cohort_summary["split_subjects"]
    for k, name in (("train", "Training"), ("val", "Val."), ("test", "Test")):
        ww = w * MIMIC_SPLIT[k]
        A.add_patch(FancyBboxPatch((xx, yb), ww, bh, boxstyle="square,pad=0", fc=SPLIT_FC[k], ec=BLUE_EC if k == "test" else GREY_EC,
                                   lw=0.5, zorder=2))
        c, narrow = xx + ww / 2, k == "val"                      # the narrow validation cell shows the numbers only
        text(A, c, yb + 0.8, name, size=4.2 if narrow else 4.7, weight="bold", ha="center")
        text(A, c, yb + 3.2, f"{100 * MIMIC_SPLIT[k]:.0f}%", size=4.2 if narrow else 4.5, ha="center")
        text(A, c, yb + 5.9, f"{ecgs[k]:,}{'' if narrow else ' ECGs'}\n{patients[k]:,}{'' if narrow else ' patients'}",
             size=4.0 if narrow else 4.4, ha="center")
        xx += ww


def models_panel(A, x, y, w):
    """c: the ten models as a grid of foundation models (rows) and adaptations (columns)."""
    lab_w, rh = 22, 6.6
    cw = (w - lab_w) / 3
    for j, (ad, lab) in enumerate((("frozen", "Frozen\n+ ridge"), ("mimic", "Fine-tuned\nMIMIC"), ("code15", "Fine-tuned\nCODE-15%"))):
        cx = x + lab_w + cw * (j + .5)
        symbol(A, cx, y + 1.8, ad, INK2, ms=3.6)
        text(A, cx, y + 3.6, lab, size=4.4, ha="center")
    y0 = y + 9.5
    A.plot([x, x + w], [y0, y0], color=GREY_EC, lw=0.5)
    text(A, x, y0 + 1.0, "Lima et al.", size=5.2, weight="bold")
    text(A, x, y0 + 3.4, "trained on CODE", size=4.3, color=INK2)
    symbol(A, x + lab_w + cw * 0.5, y0 + rh / 2, "public", FM_COLOR["Lima"], ms=4.4)
    text(A, x + lab_w + cw * 0.5 + 3, y0 + rh / 2, "used as published", size=4.6, color=INK2, va="center")
    yy = y0 + rh
    for fm in ("ECGFounder", "ECG-FM", "HuBERT-ECG"):
        A.plot([x, x + w], [yy, yy], color=GREY_EC, lw=0.4)
        text(A, x, yy + 1.0, fm, size=5.2, weight="bold", color=FM_COLOR[fm])
        for j, ad in enumerate(("frozen", "mimic", "code15")):
            symbol(A, x + lab_w + cw * (j + .5), yy + rh / 2, ad, FM_COLOR[fm], ms=4.6)
        yy += rh
    A.plot([x, x + w], [yy, yy], color=GREY_EC, lw=0.5)
    return yy


def sets_panel(A, x, y, w, sets):
    """d: the three repeat-ECG sets, each with a schematic time axis (lengths not proportional to time)."""
    rows = [("Within 48 h", "main set", sets["within_48h"], f"median {sets['within_48h']['median_hours']:.0f} h", [0, 8, 22, 41], 48,
             [(0, "0"), (48, "48 h")], True, 0.72),
            ("Within 1 h", "same visit and care setting", sets["within_1h"], f"median {sets['within_1h']['median_hours'] * 60:.0f} min",
             [0, 24], 60, [(0, "0"), (60, "1 h")], False, 0.42),
            ("Outpatient, within 30 days", "08:00–17:59", sets["outpatient_30d"],
             f"median {sets['outpatient_30d']['median_hours'] / 24:.0f} days", [0, 9, 26], 30, [(0, "0"), (30, "30 days")], False, 1.0)]
    rh, tl0 = 10.3, x + w * 0.56
    full = (x + w - 3.5) - tl0
    for i, (name, sub, s, med, times, tmax, ticks, main, frac) in enumerate(rows):
        yy = y + i * (rh + 1.0)
        box(A, x, yy, w, rh, fc=BLUE_FC if main else GREY_FC, ec=BLUE_EC if main else GREY_EC, lw=0.5)
        text(A, x + 1.8, yy + 1.2, name, size=5.2, weight="bold")
        text(A, x + 1.8 + len(name) * 1.02 + 1.2, yy + 1.35, sub, size=4.6, color=INK2)
        text(A, x + 1.8, yy + 4.4, f"{int(s['patients']):,} patients · {int(s['pairs']):,} pairs\n{med}", size=4.8)
        timeline(A, tl0, tl0 + full * frac, yy + rh - 3.0, times, tmax, BLUE_EC if main else INK2, ticks)
    return y + 3 * (rh + 1.0)


QUESTIONS = [("How much does AI-ECG age vary between repeat ECGs, and do the\ntime of day, care setting or physiology explain the variation?",
              "SEM · ICC of the age gap · variance decomposition", "Fig. 2"),
             ("Are more accurate models more reliable?", "MAE against SEM and against the age- and sex-adjusted ICC", "Fig. 3"),
             ("Does the variation weaken associations with death within\n1 year and change individual flags?",
              "Simulation–extrapolation · averaging repeat ECGs · 'older' flag", "Fig. 4")]


def questions_panel(A, x, y, w):
    """e: the three questions, with the measures and the figure that answers each."""
    rh = 10.3
    for i, (q, measures, fig_tag) in enumerate(QUESTIONS):
        yy = y + i * (rh + 1.0)
        box(A, x, yy, w, rh, fc="white", ec=GREY_EC, lw=0.5)
        A.add_patch(FancyBboxPatch((x + 1.6, yy + 1.4), 3.6, 3.6, boxstyle="circle,pad=0", fc=INK, ec="none", zorder=4))
        text(A, x + 3.4, yy + 3.2, str(i + 1), size=5.2, weight="bold", color="white", ha="center", va="center")
        text(A, x + 7.0, yy + 1.2, q, size=5.0)
        text(A, x + 7.0, yy + rh - 1.2, measures, size=4.6, color=BLUE_EC, va="bottom")
        box(A, x + w - 11.5, yy + rh / 2 - 2.2, 9.5, 4.4, fc=GREY_FC, ec=GREY_EC, lw=0.4, z=3)
        text(A, x + w - 6.75, yy + rh / 2, fig_tag, size=5.0, weight="bold", ha="center", va="center")


def draw_fig1():
    sets = {s: reliability.loc[(MODEL_KEYS[0], s)] for s in ("within_1h", "outpatient_30d", "within_48h")}
    assert all((reliability.xs(s, level="set")[["patients", "pairs"]] == sets[s][["patients", "pairs"]]).all().all() for s in sets), \
        "the repeat-ECG sets differ between models"
    or10, chrono, gaps = float(mortality.loc["lima", "or_obs"]), 63, (12, 3)
    fig, A = canvas(106)
    heading(A, 1, 1.5, "a", "Example: same patient, two ECGs 11 min apart")
    example_panel(A, or10, chrono, gaps)
    heading(A, 61, 1.5, "b", "ECGs and patients")
    cohort_panel(A, 65, 7.0, 54)
    heading(A, 121, 1.5, "c", "Ten models")
    yb = models_panel(A, 125, 7.0, 53)
    box(A, 125, yb + 1.8, 53, 9.0, fc=BLUE_FC, ec=BLUE_EC)
    text(A, 151.5, yb + 6.3, "AI-ECG age of each ECG\nage gap = AI-ECG age − chronological age", size=4.8, ha="center", va="center")
    y2 = 61
    heading(A, 1, y2 - 1.0, "d", "Repeat ECGs of the same patient")
    bottom = sets_panel(A, 5, y2 + 5.5, 82, sets)
    A.add_patch(FancyBboxPatch((3.6, y2 + 3.6), 84.8, bottom - (y2 + 3.6) + 0.6, boxstyle="square,pad=0", fc="none", ec=BLUE_EC, lw=0.6,
                               ls=(0, (3, 1.5)), zorder=1))                     # the three sets lie within the test split
    A.text(5.6, y2 + 3.6, f" Within the test split ({cohort_summary['split_subjects']['test']:,} patients) ", fontsize=4.8, color=BLUE_EC,
           ha="left", va="center", zorder=6, bbox=dict(fc="white", ec="none", pad=0.2))
    heading(A, 90, y2 - 1.0, "e", "Questions")
    questions_panel(A, 94, y2 + 5.5, 84)
    exc = cohort_summary["excluded_by_first_reason"]
    src = ([{"item": "records", "value": cohort_summary["records"]},
            {"item": "excluded_flat_or_missing_lead", "value": exc["flat_lead"] + exc["missing_lead"]},
            {"item": "excluded_age", "value": exc["age_89_or_older"] + exc["age_unavailable"] + exc["age_under_18"]},
            {"item": "included_ecgs", "value": cohort_summary["included"]},
            {"item": "included_patients", "value": cohort_summary["patients_included"]}]
           + [{"item": f"split_{k}_ecgs", "value": v} for k, v in cohort_summary["split_records"].items()]
           + [{"item": f"split_{k}_patients", "value": v} for k, v in cohort_summary["split_subjects"].items()]
           + [{"item": f"{s}_{c}", "value": int(sets[s][c])} for s in sets for c in ("patients", "pairs")]
           + [{"item": "panel_a_ages", "value": f"illustrative values {chrono + gaps[0]} and {chrono + gaps[1]} years (schematic)"}])
    save_figure(fig, "Fig1", pd.DataFrame(src), OVERWRITE)


draw_fig1()

## Fig. 2

**Fig. 2 | AI-ECG age varied widely between repeat ECGs, and time of day, care setting and physiology explained little of the
variation.** **a**, Range within which a single ECG lies, with 95% probability, around the AI-ECG age the patient would have without
retest variation (±1.96 SEM), within 48 hours (bars) and within 1 hour (ticks). The dashed line marks 8 years. **b**, Share of the
within-patient variance of AI-ECG age within 48 hours explained by differences between the two ECGs in time of day, care setting and
physiology (Shapley decomposition; 28,500 test patients), with 95% confidence intervals. **c**, Later minus earlier AI-ECG age of the same
patient within 1 hour (dashed) and within 48 hours (solid) for the Lima et al. model and the model with the lowest MAE. Pairs are weighted
so that each patient contributes equally. Frozen, frozen embedding with ridge regression; MAE, mean absolute error; MIMIC, MIMIC-IV-ECG;
QTc, corrected QT interval; SEM, standard error of measurement.

In [ ]:
def draw_fig2():
    fig = figure_mm(130)
    src = []
    top, rh = 12, 50
    # a - 95% range of one ECG around the value without retest variation
    panel_letter(fig, 1, 2, "a")
    ax = axes_mm(fig, 44, top, 50, rh)
    for i, key in enumerate(MODEL_KEYS):
        r48, r1 = reliability.loc[(key, "within_48h")], reliability.loc[(key, "within_1h")]
        v48, v1 = r48["range95"], r1["range95"]
        ax.barh(i, v48, height=0.55, color=colour(key), alpha=0.35, lw=0)
        ax.plot([v1, v1], [i - 0.33, i + 0.33], color=INK, lw=0.9)
        ax.text(v48 + 0.3, i, f"±{v48:.1f}", va="center", fontsize=5.2)
        src.append({"panel": "a", "model": key, "range95_48h": v48, "range95_1h": v1, "sem_48h": r48["sem"], "sem_1h": r1["sem"]})
    ax.axvline(8, color=INK2, lw=0.7, ls=(0, (2, 1.5)))
    model_rows(ax)
    ax.set_xlim(0, 16)
    ax.set_xticks([0, 4, 8, 12, 16])
    ax.set_xlabel("95% range of one ECG around the value\nwithout retest variation (± years, 1.96 × SEM)")
    ax.legend(handles=[Line2D([], [], color=GREY, alpha=0.35, lw=4, label="Within 48 h"),
                       Line2D([], [], color=INK, lw=0.9, label="Within 1 h, same visit"),
                       Line2D([], [], color=INK2, lw=0.7, ls=(0, (2, 1.5)), label="8 years")],
              loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3, fontsize=5.0, borderaxespad=0, handlelength=1.6, columnspacing=0.7)
    # b - Shapley shares of the within-patient variance, same rows
    panel_letter(fig, 99, 2, "b")
    unexplained = {k: 100 - decomposition.loc[(k, "all"), "share_pct"] for k in MODEL_KEYS}
    for j, (comp, title) in enumerate((("time_of_day", "Time of day"), ("care_setting", "Care setting"),
                                       ("physiology", "Heart rate, QTc,\nPR and rhythm"))):
        bx = axes_mm(fig, 104 + j * 21, top, 18, rh)
        for i, key in enumerate(MODEL_KEYS):
            d = decomposition.loc[(key, comp)]
            ci_point(bx, d["share_pct"], i, d["share_pct_lo"], d["share_pct_hi"], key)
            src.append({"panel": "b", "model": key, "component": comp, "share_pct": d["share_pct"], "lo": d["share_pct_lo"],
                        "hi": d["share_pct_hi"], "unexplained_pct": unexplained[key]})
        model_rows(bx, show=False)
        bx.set_xlim(0, 4)
        bx.set_xticks([0, 2, 4])
        bx.text(0.5, 1.02, title, transform=bx.transAxes, ha="center", va="bottom", fontsize=5.3)
        if j == 1:
            bx.set_xlabel("Share of within-patient\nvariance (%)")
    bx.text(1.1, 0.5, f"Unexplained: {min(unexplained.values()):.1f}–{max(unexplained.values()):.1f}%", transform=bx.transAxes, rotation=90,
            ha="left", va="center", fontsize=5.3, color=INK2)
    warn_if_clipped("Fig2 a-b", *fig.axes)
    # c - later minus earlier AI-ECG age: the Lima et al. model and the model with the lowest MAE (x axis from −24 to 24 years)
    panel_letter(fig, 1, 76, "c")
    best = accuracy.loc[MODEL_KEYS, "mae_mimic_test"].idxmin()
    ymax, axes = 0, []
    for j, key in enumerate(["lima", best]):
        cx = axes_mm(fig, 12 + j * 88, 80, 74, 36)
        axes.append(cx)
        for s, ls, lab in (("within_1h", (0, (2.2, 1.2)), "Within 1 h (same visit)"), ("within_48h", "-", "Within 48 h")):
            h = hist[(hist["key"] == key) & (hist["set"] == s)]
            y = h["pct"].to_numpy()
            ymax = max(ymax, np.nanmax(y))
            cx.stairs(y, np.r_[h["bin_lo"].to_numpy(), h["bin_hi"].to_numpy()[-1]], color=colour(key), lw=0.9, ls=ls, label=lab)
            src += [{"panel": "c", "model": key, "set": s, "bin_lo": a, "bin_hi": b, "share_pct": c} for a, b, c in zip(h["bin_lo"], h["bin_hi"], y)]
        s1, s48 = reliability.loc[(key, "within_1h")], reliability.loc[(key, "within_48h")]
        cx.text(0.99, 0.97, f"{model_label(key)}\nMAE {accuracy.loc[key, 'mae_mimic_test']:.1f} years\n\nDifference ≥5 years:\n"
                f"{s1['share_ge5_pct']:.0f}% within 1 h, {s48['share_ge5_pct']:.0f}% within 48 h\nDifference ≥10 years:\n"
                f"{s1['share_ge10_pct']:.0f}% within 1 h, {s48['share_ge10_pct']:.0f}% within 48 h",
                transform=cx.transAxes, ha="right", va="top", fontsize=5.3, linespacing=1.25)
        cx.set_xlim(-24, 24)
        cx.set_xticks(range(-24, 25, 4))
        cx.set_xlabel("Later minus earlier AI-ECG age of the same patient (years)")
        if j == 0:
            cx.set_ylabel("Patients (% per 2-year bin)")
            cx.legend(loc="upper left", fontsize=5.3, handlelength=2.4, borderaxespad=0.2)
        cx.axvline(0, color=GRID, lw=0.5, zorder=0)
    ytop = 4 * np.ceil(ymax * 1.1 / 4)
    for cx in axes:
        cx.set_ylim(0, ytop)
        cx.set_yticks(np.arange(0, ytop + 0.1, 4))
        cx.yaxis.set_major_formatter(FormatStrFormatter("%d"))
    model_legend(fig)
    save_figure(fig, "Fig2", pd.DataFrame(src), OVERWRITE)
    print(f"panel b: {int(decomposition.loc[(MODEL_KEYS[0], 'all'), 'patients']):,} patients")


draw_fig2()

## Fig. 3

**Fig. 3 | More accurate models were not consistently more reliable.** **a**, Mean absolute error and SEM within 48 hours. **b**, Mean
absolute error and ICC of the age gap adjusted for age and sex. Arrows connect the frozen and the MIMIC-IV-ECG-fine-tuned version of each
foundation model. Frozen, frozen embedding with ridge regression; ICC, intraclass correlation coefficient; MIMIC, MIMIC-IV-ECG; SEM,
standard error of measurement.

In [ ]:
def draw_fig3():
    fig = figure_mm(70)
    src = []
    mae = accuracy["mae_mimic_test"]
    for j, (col, ylab, ylim, name) in enumerate((("sem", "SEM within 48 h (years)", (3.5, 7.5), "sem"),
                                                 ("icc_gap_adj", "ICC of the age gap adjusted\nfor age and sex", (0.5, 0.8), "icc_adj"))):
        value = {k: reliability.loc[(k, "within_48h"), col] for k in MODEL_KEYS}
        panel_letter(fig, 20 + j * 82, 2, "ab"[j])
        ax = axes_mm(fig, 36 + j * 82, 6, 52, 46)
        for fm in ("ECGFounder", "ECG-FM", "HuBERT-ECG"):          # arrow from the frozen to the MIMIC-fine-tuned version
            a, b = [k for k in keys_of(fm) if MODELS.adapt[k] in ("frozen", "mimic")]
            ax.annotate("", xy=(mae[b], value[b]), xytext=(mae[a], value[a]),
                        arrowprops=dict(arrowstyle="-|>", color=FM_COLOR[fm], lw=0.5, shrinkA=2.5, shrinkB=2.5, mutation_scale=5, alpha=0.7))
        for key in MODEL_KEYS:
            ax.plot(mae[key], value[key], ls="none", ms=3.8, mec="white", mew=0.3, color=colour(key), marker=marker(key))
            src.append({"panel": "ab"[j], "model": key, "mae": mae[key], name: value[key]})
        ax.set_xlim(6, 11.5)
        ax.set_xticks([6, 7, 8, 9, 10, 11])
        ax.set_ylim(*ylim)
        ax.set_xlabel("Mean absolute error (years)")
        ax.set_ylabel(ylab)
        ax.grid(color=GRID, lw=0.4)
        ax.set_axisbelow(True)
    ax.text(0.97, 0.04, "Arrows: frozen →\nfine-tuned (MIMIC)", transform=ax.transAxes, ha="right", va="bottom", fontsize=5.0, color=INK2)
    warn_if_clipped("Fig3", *fig.axes)
    model_legend(fig)
    save_figure(fig, "Fig3", pd.DataFrame(src), OVERWRITE)


draw_fig3()

## Fig. 4

**Fig. 4 | The variation weakened associations with mortality and changed individual flags.** All ten models in the test split, with one
panel per foundation model in **a** and **b** (dotted lines, frozen; solid lines, fine-tuned on MIMIC-IV-ECG; dashed lines, fine-tuned on
CODE-15%). **a**, Odds ratio for death within 1 year per 10-year larger age gap of the first ECG, adjusted for age, sex and the number of
ECGs within 48 hours, when the observed within-patient variance was added again in increasing amounts (filled symbols) and extrapolated to
no variance (open symbols; 32,127 patients, 3,944 deaths). **b**, Odds ratio per 10-year larger mean age gap of the first one, two, three
and five ECGs within 7 days, adjusted for age, sex and the number of ECGs within 7 days (6,092 patients, 1,230 deaths), with 95%
confidence intervals; bars on the right show the range when each ECG was used alone. **c**, Patients whose flag for an age gap of 8 years
or more differed between the first ECG and the next ECG within 48 hours, with 95% confidence intervals. Frozen, frozen embedding with ridge
regression; MIMIC, MIMIC-IV-ECG.

In [ ]:
AVG_OFFSET = {"public": 0.0, "frozen": -0.16, "mimic": 0.0, "code15": 0.16}       # x offsets so that the CIs do not overlap
FM_OFFSET = {"Lima": 0.0, "ECGFounder": -0.05, "ECG-FM": 0.0, "HuBERT-ECG": 0.05}


def simex_panel(ax, keys, src):
    """SIMEX curves of the models `keys`: the ORs at noise multipliers 1–4 (filled symbols) and the quadratic in the log OR fitted to
    them, drawn with short dashes from 1 to 0, where it ends at the extrapolated OR (open symbol)."""
    for key in keys:
        c = simex[simex["key"] == key].sort_values("noise_multiplier")
        obs, ext = c[~c["extrapolated"].astype(bool)], c[c["extrapolated"].astype(bool)]
        coef = np.polyfit(obs["noise_multiplier"].to_numpy() - 1, np.log(obs["or_per10"].to_numpy()), 2)
        xx = np.linspace(-1, 0, 30)
        col, style = colour(key), LINE_STYLE[MODELS.adapt[key]]
        ax.plot(xx + 1, np.exp(np.polyval(coef, xx)), color=col, lw=0.68, ls=(0, (2, 1.4)))
        ax.plot(obs["noise_multiplier"], obs["or_per10"], color=col, lw=0.8, ls=style)
        ax.plot(obs["noise_multiplier"], obs["or_per10"], ls="none", ms=3.0, color=col, marker=marker(key))
        ax.plot(ext["noise_multiplier"], ext["or_per10"], ls="none", ms=3.4, color=col, marker=marker(key), mfc="white", mew=0.7)
        src += [{"panel": "a", "model": key, "noise_multiplier": a, "or_per_10y": b, "extrapolated": e}
                for a, b, e in zip(c["noise_multiplier"], c["or_per10"], c["extrapolated"])]
    ax.axhline(1, color=AXIS, lw=0.5)
    ax.axvline(1, color=GRID, lw=0.6, zorder=0)
    ax.set_xlim(-0.2, 4.2)
    ax.set_xticks([0, 1, 2, 3, 4])


def averaging_panel(ax, keys, cohort, src, panel=None):
    for key in keys:
        a = averaging[(averaging["key"] == key) & (averaging["cohort"] == cohort)]
        mean, single = a[a["kind"] == "mean"].sort_values("k"), a[a["kind"] == "single"]
        off = AVG_OFFSET[MODELS.adapt[key]] + FM_OFFSET[MODELS.fm[key]]
        xs = np.array([{1: 1, 2: 2, 3: 3, 5: 4}[k] for k in mean["k"]]) + off        # 1, 2, 3 and 5 ECGs at evenly spaced positions
        o, lo, hi = mean["or_per10"], mean["or_per10_lo"], mean["or_per10_hi"]
        col = colour(key)
        ax.plot(xs, o, color=col, lw=0.7, ls=LINE_STYLE[MODELS.adapt[key]])
        ax.errorbar(xs, o, yerr=[o - lo, hi - o], fmt="none", ecolor=col, elinewidth=0.5, alpha=0.6)
        ax.plot(xs, o, ls="none", marker=marker(key), ms=2.8, color=col)
        x5 = 5.2 + off * 1.3
        ax.plot([x5, x5], [single["or_per10"].min(), single["or_per10"].max()], color=col, lw=1.8, solid_capstyle="butt", alpha=0.55)
        p = {"panel": panel} if panel else {}
        src += [{**p, "model": key, "kind": "mean", "k": k, "or": v, "lo": l, "hi": h} for k, v, l, h in zip(mean["k"], o, lo, hi)]
        src += [{**p, "model": key, "kind": "single", "k": k, "or": v} for k, v in zip(single["k"], single["or_per10"])]
    ax.axhline(1, color=AXIS, lw=0.5)
    ax.axvline(4.6, color=GRID, lw=0.6)
    ax.set_xticks([1, 2, 3, 4, 5.2])
    ax.set_xticklabels(["1", "2", "3", "5", "Each\nalone"])
    ax.set_xlim(0.5, 5.8)


def flags_panel(cx, src):
    for i, key in enumerate(MODEL_KEYS):
        r = flags.loc[key]
        ci_point(cx, r["differs_pct"], i, r["differs_pct_lo"], r["differs_pct_hi"], key)
        src.append({"panel": "c", "model": key, "flag_differs_pct": r["differs_pct"], "lo": r["differs_pct_lo"], "hi": r["differs_pct_hi"]})
    model_rows(cx)
    cx.set_xlim(0, 25)
    cx.set_xticks([0, 5, 10, 15, 20, 25])
    cx.set_xlabel("Patients whose 'older' flag\nchanged within 48 h (%)")


def odds_ratio_figure(name, cohort, simex_row, height, ytop_simex=1.40, ytop_avg=1.50):
    """Small multiples, one panel per foundation model: SIMEX curves (row a, optional) and averaging (row b); flags (c) with row a."""
    fig = figure_mm(height)
    src = []
    x0, gap, w, h = 16, 3.5, 38.5, 38
    for letter, y0, kind in ((("a", 6, "simex"), ("b", 58, "averaging")) if simex_row else ((None, 6, "averaging"),)):
        if letter:
            panel_letter(fig, 1, y0 - 4, letter)
        for j, (fm, title) in enumerate(FM_PANELS):
            ax = axes_mm(fig, x0 + j * (w + gap), y0, w, h)
            if kind == "simex":
                simex_panel(ax, keys_of(fm), src)
                ax.set_xticklabels(["0", "1", "2", "3", "4"])
                or_axis(ax, 0.95, ytop_simex, 0.1, label=(j == 0))
                if j == 0:
                    ax.text(0.0, -0.25, "0 = no noise, 1 = observed", transform=ax.transAxes, fontsize=5.0, color=INK2)
                if j == 1:
                    ax.set_xlabel("Retest noise variance (× observed)", x=1.05)
            else:
                averaging_panel(ax, keys_of(fm), cohort, src, letter)
                or_axis(ax, 0.80, ytop_avg, 0.2, label=(j == 0))
                if j == 1:
                    ax.set_xlabel(f"ECGs averaged (first 5 ECGs within {'7 days' if cohort == 'within_7d' else '48 h'})", x=1.05)
            panel_title(ax, fm, title)
    if simex_row:
        panel_letter(fig, 1, 106, "c")
        flags_panel(axes_mm(fig, 50, 110, 40, 40), src)
    warn_if_clipped(name, *fig.axes)
    model_legend(fig, y_mm=1.5)
    save_figure(fig, name, pd.DataFrame(src), OVERWRITE)


odds_ratio_figure("Fig4", "within_7d", simex_row=True, height=170, ytop_simex=1.40, ytop_avg=1.50)
r = mortality.loc["lima"]
print(f"a: {int(r['patients']):,} patients, {int(r['deaths']):,} deaths; "
      f"b: {int(r['avg7d_patients']):,} patients, {int(r['avg7d_deaths']):,} deaths")

## Supplementary Fig. 1

**Supplementary Fig. 1 | Distribution of the within-patient difference in AI-ECG age for all ten models.** Later minus earlier AI-ECG age
of the same patient for ECGs within 1 hour at the same visit (dashed; 7,404 patients) and within 48 hours (solid; 32,127 patients), in
2-year bins shown from −24 to 24 years; bins with fewer than 10 patients are not shown. The
percentages are the shares of pairs that differed by at least 5 and at least 10 years (within 1 hour / within 48 hours).

In [ ]:
def draw_supp_fig1():
    fig = figure_mm(112)
    src = []
    top = 8 * np.ceil(hist["pct"].max() * 1.6 / 8)                  # the same y axis in every panel, with room for the annotation
    for i, key in enumerate(MODEL_KEYS):
        x0, y0 = (14, 6) if i == 0 else (68 + (i - 1) % 3 * 38, 6 + (i - 1) // 3 * 33)    # Lima et al. model alone, then 3 x 3
        ax = axes_mm(fig, x0, y0, 32, 22)
        for s, ls in (("within_1h", (0, (2.2, 1.2))), ("within_48h", "-")):
            h = hist[(hist["key"] == key) & (hist["set"] == s)]
            y = h["pct"].to_numpy()
            ax.stairs(y, np.r_[h["bin_lo"].to_numpy(), h["bin_hi"].to_numpy()[-1]], color=colour(key), lw=0.8, ls=ls)
            src += [{"model": key, "set": s, "bin_lo": a, "bin_hi": b, "share_pct": c} for a, b, c in zip(h["bin_lo"], h["bin_hi"], y)]
        s1, s48 = reliability.loc[(key, "within_1h")], reliability.loc[(key, "within_48h")]
        ax.text(0.02, 0.97, model_label(key), transform=ax.transAxes, ha="left", va="top", fontsize=5.2, fontweight="bold")
        ax.text(0.02, 0.84, f"≥5 y: {s1['share_ge5_pct']:.0f}% / {s48['share_ge5_pct']:.0f}%\n"
                f"≥10 y: {s1['share_ge10_pct']:.0f}% / {s48['share_ge10_pct']:.0f}%",
                transform=ax.transAxes, ha="left", va="top", fontsize=4.8, linespacing=1.2, color=INK2)
        src += [{"model": key, "set": s, "abs_ge5_pct": v["share_ge5_pct"], "abs_ge10_pct": v["share_ge10_pct"]}
                for s, v in (("within_1h", s1), ("within_48h", s48))]
        ax.set_xlim(-24, 24)
        ax.set_xticks([-24, -12, 0, 12, 24])
        ax.set_ylim(0, top)
        ax.set_yticks(np.arange(0, top + 0.1, 8))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%d"))
        ax.axvline(0, color=GRID, lw=0.5, zorder=0)
        if i == 0 or (i - 1) % 3 == 0:
            ax.set_ylabel("Patients (% per\n2-year bin)")
        if i == 0 or i >= 7:
            ax.set_xlabel("Later minus earlier\nAI-ECG age (years)")
    fig.text(14 / FIGURE_WIDTH, 1 - 40 / 112, "Dashed: within 1 h, same visit\nSolid: within 48 h", fontsize=5.3, va="top", linespacing=1.3)
    model_legend(fig)
    save_figure(fig, "SuppFig1", pd.DataFrame(src), OVERWRITE)
    print(f"within 1 h {int(reliability.loc[('lima', 'within_1h'), 'patients']):,} patients, within 48 h "
          f"{int(reliability.loc[('lima', 'within_48h'), 'patients']):,} patients")


draw_supp_fig1()

## Supplementary Fig. 2

**Supplementary Fig. 2 | Standard error of measurement by repeat ECG set and care setting.** **a**, Pairs within 1 hour at the same visit;
the same pairs restricted to one recording machine and to ECGs whose machine report did not mention artifact, noise, non-standard or
additional leads, or lead reversal; outpatient pairs within 30 days; and all pairs within 48 hours. **b**, Pairs within 48 hours with
both ECGs in the emergency department, on a ward or in an intensive care unit, and all pairs within 48 hours. SEM, standard error of
measurement.

In [ ]:
SUPP_FIG2_SETS = {   # set, legend label, grey marker (None: the model's own colour and marker)
    "a": [("within_1h", "Within 1 h, same visit", dict(marker="v", ms=3.0, mew=0.7)),
          ("within_1h_same_machine_no_flag", "Within 1 h, same machine, no quality statement", dict(marker="v", ms=3.0, mew=0.7, mfc="white")),
          ("outpatient_30d", "Outpatient, within 30 days", dict(marker="x", ms=3.2, mew=0.8)),
          ("within_48h", "Within 48 h", None)],
    "b": [("within_48h_ed", "Both in the emergency department", dict(marker="P", ms=3.6, mew=0)),
          ("within_48h_ward", "Both on a ward", dict(marker="d", ms=3.6, mew=0.3)),
          ("within_48h_icu", "Both in an intensive care unit", dict(marker="*", ms=4.6, mew=0.3)),
          ("within_48h", "All pairs within 48 h", None)]}


def colour_handle():
    """Legend entry for 'the model's own colour and marker': the four colours side by side."""
    return tuple(Line2D([], [], ls="none", marker=mk, color=FM_COLOR[f], ms=2.6)
                 for f, mk in (("Lima", MARKER["public"]), ("ECGFounder", "o"), ("ECG-FM", "o"), ("HuBERT-ECG", "o")))


def draw_supp_fig2():
    top, rh = 17, 62
    fig = figure_mm(top + rh + 16)
    src = []
    offsets = [-0.3, -0.1, 0.1, 0.3]                                  # sub-rows of each model row, in the order of the legend
    grey = dict(ls="none", color=GREY, mec=GREY, zorder=3)
    for p, (x0, lx) in zip("ab", ((44, 1), (112, 104))):
        panel_letter(fig, lx, 2, p)
        ax = axes_mm(fig, x0, top, 54, rh)
        for i, key in enumerate(MODEL_KEYS):
            for (s, lab, style), dy in zip(SUPP_FIG2_SETS[p], offsets):
                r = reliability.loc[(key, s)]
                if style is None:
                    ax.plot(r["sem"], i + dy, ls="none", marker=marker(key), ms=3.2, color=colour(key), zorder=4)
                else:
                    ax.plot(r["sem"], i + dy, **grey, **style)
                src.append({"panel": p, "model": key, "set": s, "label": lab, "sem": r["sem"], "patients": r["patients"], "pairs": r["pairs"]})
        model_rows(ax, show=(p == "a"), separators=range(1, 10))
        ax.set_xlim(3, 9)
        ax.set_xticks(range(3, 10))
        ax.set_xlabel("SEM (years)")
        handles = [Line2D([], [], **grey, **style) if style else colour_handle() for _, _, style in SUPP_FIG2_SETS[p]]
        ax.legend(handles, [lab for _, lab, _ in SUPP_FIG2_SETS[p]], loc="lower left", bbox_to_anchor=(0, 1.02), fontsize=5.3,
                  handletextpad=0.5, borderaxespad=0, labelspacing=0.35, handlelength=2.6,
                  handler_map={tuple: HandlerTuple(ndivide=None, pad=0.15)})
    warn_if_clipped("SuppFig2", *fig.axes)
    model_legend(fig)
    save_figure(fig, "SuppFig2", pd.DataFrame(src), OVERWRITE)


draw_supp_fig2()

## Supplementary Fig. 3

**Supplementary Fig. 3 | Regression coefficient for the night-minus-daytime difference in AI-ECG age.** Pairs of ECGs of the same patient
within 48 hours, one at night (00:00–05:59) and one in the daytime (10:00–15:59) (5,697 patients). Colored symbols are regression
intercepts, that is, the expected difference in AI-ECG age between a night and a daytime ECG of the same patient recorded in the same care
setting, with 95% confidence intervals; positive values indicate an older AI-ECG age at night, and filled symbols indicate intervals that
exclude zero. The shaded band marks ±1 year for reference.

In [ ]:
def direction_labels(ax, left, right, offset_mm=9.0):
    """'← left | right →' under the x axis, split at zero."""
    y = -offset_mm / (ax.get_window_extent().height / ax.figure.dpi / MM)
    tr = ax.get_xaxis_transform()
    ax.text(0, y, f"← {left}  ", transform=tr, ha="right", va="top", fontsize=5.5, color=INK2, clip_on=False)
    ax.text(0, y, f"  {right} →", transform=tr, ha="left", va="top", fontsize=5.5, color=INK2, clip_on=False)


def draw_supp_fig3():
    row_mm, ylo, yhi = 6.6, -0.95, 9.85
    ax_h = (yhi - ylo) * row_mm
    fig = figure_mm(8 + ax_h + 20)
    ax = axes_mm(fig, 46, 8, 86, ax_h)
    ax.axvspan(-1, 1, color=BAND, zorder=0, lw=0)
    ax.axvline(0, color=AXIS, lw=0.5, zorder=1)
    src = []
    for i, key in enumerate(MODEL_KEYS):
        r = night_day.loc[key]
        filled = bool(r["estimate_lo"] > 0 or r["estimate_hi"] < 0)     # filled when the 95% CI excludes zero
        ci_point(ax, r["estimate"], i, r["estimate_lo"], r["estimate_hi"], key, filled)
        src.append({"model": key, "patients": r["patients"], "pairs": r["pairs"], "night_minus_day": r["estimate"],
                    "lo": r["estimate_lo"], "hi": r["estimate_hi"], "ci_excludes_zero": filled})
    for j in range(9):
        ax.axhline(j + 0.83, color=GRID, lw=0.5, zorder=0)
    ax.set_yticks(range(len(MODEL_KEYS)))
    ax.set_yticklabels(LABELS)
    ax.tick_params(axis="y", length=0)
    ax.set_ylim(yhi, ylo)
    ax.set_xlim(-1.5, 1.5)
    ax.set_xticks([-1.5, -1, -0.5, 0, 0.5, 1, 1.5])
    ax.set_xlabel("Night minus daytime AI-ECG age (years)")
    direction_labels(ax, "Older by day", "Older at night", offset_mm=8.5)
    warn_if_clipped("SuppFig3", ax)
    model_legend(fig)
    save_figure(fig, "SuppFig3", pd.DataFrame(src), OVERWRITE)
    print(f"{int(night_day['patients'].iloc[0]):,} patients; CI excludes zero in {sum(s['ci_excludes_zero'] for s in src)} models")


draw_supp_fig3()

## Supplementary Fig. 4

**Supplementary Fig. 4 | Why the ICC of the age gap fell after fine-tuning although the SEM did not.** Measures. The age gap is AI-ECG age
minus chronological age. The within-patient variation shows how much the age gap of one patient changes between repeat ECGs. Its standard
deviation is the SEM, and its square is the within-patient variance. The between-patient variation shows how much the age gaps of
different patients differ, apart from this retest variation. Its standard deviation is the between-patient SD in Table 2, and its square
is the between-patient variance. The ICC of the age gap is a reliability index from 0 to 1 that shows how consistently the age gap comes
out between repeat ECGs of the same patient, relative to how much it differs between patients. It is the between-patient variance divided
by the total variance and approaches 1 when patients differ much more than repeat ECGs of one patient. **a**, Concept (illustrative
values). Each curve shows the age gaps from repeat ECGs of one patient. Its centre is the patient's age gap without retest variation, and
its width is the within-patient variation. The centres of the three patients are far apart in the upper row and closer together in the
lower row, and every curve has the same width. When the centres move closer and the width stays the same, the curves overlap more, a
single ECG separates patients less well, and the ICC is lower. **b**, Variance of the age gap (years<sup>2</sup>) for each model in the
pairs within 48 hours of the test split (32,127 patients, each weighted equally). The dark part of each bar is the between-patient
variance and the light part the within-patient variance, so the whole bar is the total variance. The number at its end is the ICC, the
dark part as a share of the bar (for the Lima et al. model, 111.1 ÷ 149.0 = 0.745). Fine-tuning on MIMIC-IV-ECG shortened the dark part
but changed the light part little. The ICC therefore fell mainly because the age gaps of different patients became more alike, and only
slightly because single ECGs varied more. ICC, intraclass correlation coefficient; SD, standard deviation; SEM, standard error of
measurement.

In [ ]:
def tint(hex_colour, f=0.55):
    """The colour mixed with white (share f of white)."""
    c = np.array([int(hex_colour[i:i + 2], 16) for i in (1, 3, 5)]) / 255
    return tuple(c + (1 - c) * f)


def draw_supp_fig4():
    r48 = reliability.xs("within_48h", level="set").loc[MODEL_KEYS]
    v = pd.DataFrame({"key": MODEL_KEYS, "fm": MODELS.fm[MODEL_KEYS].to_numpy(), "adapt": MODELS.adapt[MODEL_KEYS].to_numpy(),
                      "between_sd": r48["between_sd"].to_numpy(), "sem": r48["sem"].to_numpy(), "icc_gap": r48["icc_gap"].to_numpy()})
    v["between_var"], v["within_var"] = v["between_sd"] ** 2, v["sem"] ** 2
    fig = figure_mm(92)
    # a - concept: three patients, each with repeat ECGs scattered around the patient's own value (illustrative values)
    panel_letter(fig, 1, 3, "a")
    x = np.linspace(-26, 26, 600)
    for i, (title, spread) in enumerate((("Wider spread between patients", 11.0), ("Narrower spread between patients", 7.5))):
        ax = axes_mm(fig, 7, 9 + i * 36, 52, 24)
        for j, c in enumerate((-1, 0, 1)):
            y = np.exp(-0.5 * ((x - c * spread) / 4.4) ** 2)
            ax.fill_between(x, y, color=tint("#2a78d6", 0.35 + 0.2 * j), lw=0)
            ax.plot(x, y, color="#2a78d6", lw=0.6)
        ax.set_xlim(-26, 26)
        ax.set_ylim(0, 1.35)
        ax.set_yticks([])
        ax.spines["left"].set_visible(False)
        ax.set_xticks([-20, -10, 0, 10, 20])
        ax.set_xlabel("Age gap (years)" if i == 1 else "")
        ax.text(0.0, 1.10, title, transform=ax.transAxes, fontsize=5.8, fontweight="bold")
        ax.annotate("", xy=(spread, 1.12), xytext=(-spread, 1.12), arrowprops=dict(arrowstyle="<->", lw=0.6, color=INK))
        ax.text(0, 1.18, "between patients", ha="center", fontsize=5.0, color=INK2)
        ax.annotate("", xy=(4.4, 0.62), xytext=(-4.4, 0.62), arrowprops=dict(arrowstyle="<->", lw=0.5, color=INK2))
        ax.text(0, 0.68, "within", ha="center", fontsize=4.6, color=INK2)
        ax.text(1.0, 0.78, "same within-patient\nspread (SEM),\nlower ICC" if i == 1 else "higher ICC", transform=ax.transAxes,
                ha="right", va="top", fontsize=4.8, color=INK2)
    fig.text(7 / FIGURE_WIDTH, 1 - 84 / 92, "Each curve: repeat ECGs of one patient.\nICC = between-patient variance ÷ (between + within)",
             fontsize=4.8, color=INK2, va="top")
    # b - variance of the age gap split into between and within patients
    panel_letter(fig, 66, 3, "b")
    ax = axes_mm(fig, 122, 8, 50, 70)
    for k, r in enumerate(v.itertuples()):
        col = FM_COLOR[r.fm]
        ax.barh(k, r.between_var, height=0.62, color=col, lw=0)
        ax.barh(k, r.within_var, left=r.between_var, height=0.62, color=tint(col, 0.7), lw=0)
        ax.text(r.between_var + r.within_var + 2, k, f"{r.icc_gap:.2f}", va="center", fontsize=5.2)
    model_rows(ax)
    ax.set_xlim(0, 175)
    ax.set_xticks([0, 50, 100, 150])
    ax.set_xlabel("Variance of the age gap (years²)")
    ax.text(1.0, 1.02, "ICC", transform=ax.transAxes, ha="right", fontsize=5.4, fontweight="bold")
    warn_if_clipped("SuppFig4 b", ax)
    fig.legend(handles=[Patch(color=GREY, label="Between-patient variance (SD²)"), Patch(color=tint(GREY, 0.7), label="Within-patient variance (SEM²)")],
               loc="lower center", bbox_to_anchor=(0.55, 1 / 92), ncol=2, fontsize=5.2, handlelength=1.2, columnspacing=1.2, frameon=False)
    fig.text(0.865, 3.3 / 92, "ICC = dark share of each bar", ha="center", va="center", fontsize=5.2, color=INK2)
    save_figure(fig, "SuppFig4", v, OVERWRITE)
    lima = v.iloc[0]
    print(f"Lima et al. model: {lima['between_var']:.1f} ÷ {lima['between_var'] + lima['within_var']:.1f} = {lima['icc_gap']:.3f}")


draw_supp_fig4()

## Supplementary Fig. 5

**Supplementary Fig. 5 | Pull of AI-ECG age toward the mean age.** **a**, Mean AI-ECG age by chronological age in the test split, in 5-year
age bins (18–24 years for the first bin and 85–88 years for the last), with one panel per foundation model (dotted lines, frozen; solid
lines, fine-tuned on MIMIC-IV-ECG; dashed lines, fine-tuned on CODE-15%). The dashed grey line marks equality of AI-ECG age and
chronological age. A curve flatter than this line means that younger patients receive older estimates and older patients younger ones,
that is, that the estimates are pulled toward the mean age. **b**, Slope of AI-ECG age on chronological age from weighted linear
regression, with 95% confidence intervals from patient bootstrap resampling. A slope of 1 means no pull, and smaller values a stronger
pull. ECGs are weighted so that each patient counts equally.

In [ ]:
def draw_supp_fig5():
    fig = figure_mm(118)
    src = []
    x0, gap, w, h = 16, 3.5, 38.5, 38
    panel_letter(fig, 1, 2, "a")
    for j, (fm, title) in enumerate(FM_PANELS):
        ax = axes_mm(fig, x0 + j * (w + gap), 6, w, h)
        ax.plot([15, 90], [15, 90], color=AXIS, lw=0.6, ls=(0, (3, 2)), zorder=0)       # AI-ECG age = chronological age
        for key in keys_of(fm):
            b = slope_by_age[(slope_by_age["key"] == key) & slope_by_age["mean_ai_ecg_age"].notna()]
            ax.plot(b["mean_age"], b["mean_ai_ecg_age"], color=colour(key), lw=0.8, ls=LINE_STYLE[MODELS.adapt[key]])
            ax.plot(b["mean_age"], b["mean_ai_ecg_age"], ls="none", marker=marker(key), ms=2.4, color=colour(key))
            src += [{"panel": "a", "model": key, "age_bin": r.age_bin, "mean_age": r.mean_age, "mean_ai_ecg_age": r.mean_ai_ecg_age,
                     "mean_gap": r.mean_gap, "patients": r.patients} for r in b.itertuples()]
        ax.set_xlim(15, 90)
        ax.set_xticks([20, 40, 60, 80])
        ax.set_ylim(15, 90)
        ax.set_yticks([20, 40, 60, 80])
        if j == 0:
            ax.set_ylabel("Mean AI-ECG age (years)")
            ax.text(0.97, 0.06, "Dashed line:\nAI-ECG age =\nchronological age", transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=4.6, color=INK2)
        else:
            ax.set_yticklabels([])
        if j == 1:
            ax.set_xlabel("Chronological age (years)", x=1.05)
        panel_title(ax, fm, title)
    panel_letter(fig, 1, 56, "b")
    bx = axes_mm(fig, 50, 60, 56, 44)
    for i, key in enumerate(MODEL_KEYS):
        s = slope.loc[key]
        ci_point(bx, s["slope"], i, s["slope_lo"], s["slope_hi"], key)
        bx.text(1.035, i, f"{s['slope']:.2f}", va="center", fontsize=5.0)
        src.append({"panel": "b", "model": key, "slope": s["slope"], "lo": s["slope_lo"], "hi": s["slope_hi"]})
    model_rows(bx)
    bx.axvline(1, color=AXIS, lw=0.5)
    bx.set_xlim(0.3, 1.1)
    bx.set_xticks([0.4, 0.6, 0.8, 1.0])
    bx.set_xlabel("Slope of AI-ECG age on chronological age")
    bx.text(1.0, -0.9, "1 = no pull toward\nthe mean age", ha="right", va="bottom", fontsize=4.6, color=INK2)
    warn_if_clipped("SuppFig5", *fig.axes)
    model_legend(fig, y_mm=1.5)
    save_figure(fig, "SuppFig5", pd.DataFrame(src), OVERWRITE)


draw_supp_fig5()

## Supplementary Fig. 6

**Supplementary Fig. 6 | Averaging the age gap over five ECGs within 48 hours.** Odds ratio for death within 1 year per 10-year larger
mean age gap of the first one, two, three and five ECGs, adjusted for age, sex and the number of ECGs within 48 hours, in 2,711 test
patients with five or more ECGs within 48 hours (464 deaths), with one panel per foundation model (dotted lines, frozen; solid lines,
fine-tuned on MIMIC-IV-ECG; dashed lines, fine-tuned on CODE-15%) and 95% confidence intervals. Bars on the right show the range when each
of the five ECGs was used alone.

In [ ]:
odds_ratio_figure("SuppFig6", "within_48h", simex_row=False, height=62, ytop_avg=2.00)
r = mortality.loc["lima"]
print(f"{int(r['avg48h_patients']):,} patients, {int(r['avg48h_deaths']):,} deaths")